In [1]:
"""
================================================================================
Generate Baseline Permanent Water Raster (1984+) per Tile
- Extracts and stacks JRC Yearly Water History bands.
- Applies temporal gap-filling: if a pixel has NoData in the earliest year, 
  it looks ahead to subsequent years until a valid observation is found.
- Binarizes the result: Permanent Water (class 3) -> 1; everything else -> 0.
================================================================================
"""

from pathlib import Path
import numpy as np
import rasterio

# ==========================================
# 1. PATHS & CONFIGURATION
# ==========================================
JRC_REPROJ_DIR = Path(r"D:\MANGROVES\JRC_GSW\JRC_GSW1_4_YearlyHistory\reprojected_54034")
OUT_BASELINE_DIR = Path(r"D:\MANGROVES\Research_Article\Data\GSW_1984_baseline")
OUT_BASELINE_DIR.mkdir(parents=True, exist_ok=True)

# Find all reprojected JRC rasters
raster_files = list(JRC_REPROJ_DIR.glob("*_54034.tif"))
print(f"🚀 Found {len(raster_files)} reprojected JRC raster(s) to process.")

# ==========================================
# 2. PROCESSING LOOP PER TILE
# ==========================================
for i, src_path in enumerate(raster_files, 1):
    out_path = OUT_BASELINE_DIR / f"{src_path.stem.replace('_54034', '')}_baseline_water.tif"
    
    if out_path.exists():
        continue  # Skip if already processed

    print(f"[{i}/{len(raster_files)}] ⚙️ Processing baseline water for: {src_path.name} ...")

    try:
        with rasterio.open(src_path) as src:
            meta = src.meta.copy()
            
            # Read all bands (each band represents a sequential year)
            # Typical shape: (nb_bands, height, width)
            data_stack = src.read()
            nb_bands, height, width = data_stack.shape
            
            # Initialize output baseline array of the size of a single band, filled with zeros (Land)
            baseline_raster = np.zeros((height, width), dtype=np.uint8)
            
            # Global mask to track pixels that have already been resolved across years
            resolved_mask = np.zeros((height, width), dtype=bool)
            
            # Iterate through bands chronologically from oldest to newest (index 0 to nb_bands - 1)
            for b_idx in range(nb_bands):
                # If all pixels are already resolved, stop early
                if np.all(resolved_mask):
                    break
                    
                band_data = data_stack[b_idx, :, :]
                
                # Define valid pixels (in JRC GSW, valid classes are 1, 2, 3. Excluding 0 or nodata).
                valid_pixels_mask = (band_data > 0) & (~resolved_mask)
                
                if not np.any(valid_pixels_mask):
                    continue
                
                # Among valid pixels, identify Permanent Water (value == 3)
                permanent_water_mask = valid_pixels_mask & (band_data == 3)
                
                # Assign 1 where permanent water occurs
                baseline_raster[permanent_water_mask] = 1
                # (Valid pixels that are not permanent water remain 0 due to initial zeros)
                
                # Mark these pixels as resolved so they won't be overwritten by subsequent years
                resolved_mask[valid_pixels_mask] = True

            # Update metadata for the single-band output raster
            meta.update({
                "count": 1,
                "dtype": "uint8",
                "nodata": None,  # No nodata since non-water elements are classified as 0 (land)
                "compress": "lzw"
            })

            # Write the final baseline raster
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(baseline_raster, 1)

        print(f"  ✅ Saved baseline raster: {out_path.name}")

    except Exception as e:
        print(f"  ❌ Error processing {src_path.name}: {e}")

print(f"\n🎉 Baseline permanent water generation complete! Saved in: {OUT_BASELINE_DIR}")

🚀 Found 1261 reprojected JRC raster(s) to process.
[1/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E008_54034.tif ...
  ✅ Saved baseline raster: JRC_Yearly_N00E008_baseline_water.tif
[2/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E009_54034.tif ...
  ✅ Saved baseline raster: JRC_Yearly_N00E009_baseline_water.tif
[3/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E041_54034.tif ...
  ✅ Saved baseline raster: JRC_Yearly_N00E041_baseline_water.tif
[4/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E042_54034.tif ...
  ✅ Saved baseline raster: JRC_Yearly_N00E042_baseline_water.tif
[5/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E073_54034.tif ...
  ✅ Saved baseline raster: JRC_Yearly_N00E073_baseline_water.tif
[6/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E097_54034.tif ...
  ✅ Saved baseline raster: JRC_Yearly_N00E097_baseline_water.tif
[7/1261] ⚙️ Processing baseline water for: JRC_Yearly_N00E098_54034.tif ...
  ✅ Saved baseline raster: 